In [24]:
#Nombre: Cristina Maureira  RUT: 22.356.447-k
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import os
import pickle
import csv

In [45]:
class CatalogoEstelar:
    def __init__(self, ruta_csv):
        self.ruta_csv = ruta_csv
        # Listas temporales para almacenar las columnas
        ids = []
        nombres = []
        temperaturas = []
        luminosidades = []
        distancias = []
        # Lectura del archivo CSV
        with open(self.ruta_csv, mode='r', encoding='utf-8') as archivo:
            lector = csv.DictReader(archivo)
            for fila in lector:
                ids.append(fila['id'])
                nombres.append(fila['nombre'])
                # Convertimos a float para manejar valores numéricos y NaNs
                temperaturas.append(float(fila['temperatura_K']) if fila['temperatura_K'] else np.nan)
                luminosidades.append(float(fila['luminosidad_Lsol']) if fila['luminosidad_Lsol'] else np.nan)
                distancias.append(float(fila['distancia_pc']) if fila['distancia_pc'] else np.nan)
        # Guardar como arreglos independientes de NumPy
        self.ids = np.array(ids)
        self.nombres = np.array(nombres)
        self.temperaturas = np.array(temperaturas)
        self.luminosidades = np.array(luminosidades)
        self.distancias = np.array(distancias)
        # Ejecutar el método de limpieza
        self.limpiar_datos()

    def limpiar_datos(self):
        # Valores ausentes o no finitos en columnas numéricas
        no_finitos = (~np.isfinite(self.temperaturas)) | \
                     (~np.isfinite(self.luminosidades)) | \
                     (~np.isfinite(self.distancias))
        
        # Valores no positivos en propiedades físicas (temperatura, luminosidad, distancia)
        no_positivos = (self.temperaturas <= 0) | \
                       (self.luminosidades <= 0) | \
                       (self.distancias <= 0)
        # Buscamos los índices de la primera aparición de cada ID único
        _, indices_unicos = np.unique(self.ids, return_index=True)
        # Creamos una máscara de booleanos para los duplicados (True si es duplicado)
        duplicados = np.ones(len(self.ids), dtype=bool)
        duplicados[indices_unicos] = False  # Las primeras apariciones no son duplicadas
        # Combinamos todos los criterios de invalidez
        invalido = no_finitos | no_positivos | duplicados
        # Eliminar los registros inválidos aplicando los cambios a todas las columnas
        # La máscara para mantener registros válidos es la inversa de 'es_invalido'
        valido = ~invalido
        self.ids = self.ids[valido]
        self.nombres = self.nombres[valido]
        self.temperaturas = self.temperaturas[valido]
        self.luminosidades = self.luminosidades[valido]
        self.distancias = self.distancias[valido]

    def resumen(self):
        # Se obtienen los nombres con los índices de la estrella del valor máximo de temperatura
        nombre_mas_caliente = self.nombres[np.argmax(self.temperaturas)]
        nombre_mas_cercana = self.nombres[np.argmin(self.distancias)]
        # Print de los resultados
        # Utilizé Numpy dentro del print para hacer todo más limpio
        print(f"""=================RESUMEN DEL CATÁLOGO ESTELAR=================            
        -------------------Propiedades Generales----------------------
        Número total de estrellas: {len(self.ids)}
        Temperatura media: {np.mean(self.temperaturas):.2f} K
        Temperatura máxima: {np.max(self.temperaturas):.2f} K
        Luminosidad mediana: {np.median(self.luminosidades):.2f} L☉
        Distancia mínima: {np.min(self.distancias):.2f} pc
        Distancia máxima: {np.max(self.distancias):.2f} pc
        ----------------------Estrellas Clave-------------------------
        Estrella de mayor temperatura: 
            Nombre: "{nombre_mas_caliente}"
            Temperatura: {np.max(self.temperaturas):.2f} K
        Estrella más cercana: 
            Nombre: "{nombre_mas_cercana}"
            Distancia: {np.min(self.distancias):.2f} pc""")

    def _asegurar_directorio(self):
        # Método auxiliar para crear la carpeta resultados/ si no existe
        os.makedirs("resultados", exist_ok=True)

    # Formato CSV
    def guardar_csv(self):
        # Guarda todas las columnas en resultados/catalogo_copia.csv con encabezados
        self._asegurar_directorio()
        ruta = os.path.join("resultados", "catalogo_copia.csv")
        with open(ruta, mode='w', encoding='utf-8', newline='') as archivo:
            escritor = csv.writer(archivo)
            # Escribir encabezados
            escritor.writerow(['id', 'nombre', 'temperatura_K', 'luminosidad_Lsol', 'distancia_pc'])
            # Escribir filas combinando los arreglos
            for i in range(len(self.ids)):
                escritor.writerow([
                    self.ids[i],
                    self.nombres[i]
                    self.temperaturas[i], 
                    self.luminosidades[i], 
                    self.distancias[i]
                ])
        print(f"Catálogo guardado en CSV con éxito: {ruta}")
    # Formato NPY (Solo Temperaturas)
    def guardar_npy(self):
        # Guarda el arreglo de temperaturas en resultados/temperaturas.npy
        self._asegurar_directorio()
        ruta = os.path.join("resultados", "temperaturas.npy")
        np.save(ruta, self.temperaturas)
        print(f"Temperaturas guardadas en NPY: {ruta}")
    def cargar_npy(self):
        # Recupera las temperaturas desde el archivo NPY
        ruta = os.path.join("resultados", "temperaturas.npy")
        temperaturas_cargadas = np.load(ruta)
        return temperaturas_cargadas

    # Formato NPZ
    def guardar_npz(self):
        # Guarda todas las columnas comprimidas en resultados/catalogo_completo.npz
        self._asegurar_directorio()
        ruta = os.path.join("resultados", "catalogo_completo.npz")
        # Al ser self.ids un arreglo nativo de texto de NumPy, se guarda automáticamente
        # sin necesidad de habilitar pickle.
        np.savez_compressed(
            ruta, 
            ids=self.ids,
            nombres=self.nombres
            temperaturas=self.temperaturas, 
            luminosidades=self.luminosidades, 
            distancias=self.distancias
        )
        print(f"Catálogo completo guardado en NPZ: {ruta}")

    def cargar_npz(self):
        # Recupera todas las columnas desde el archivo NPZ
        ruta = os.path.join("resultados", "catalogo_completo.npz")
        with np.load(ruta, allow_pickle=False) as datos:
            self.ids = datos['ids']
            self.nombres = datos['nombres']
            self.temperaturas = datos['temperaturas']
            self.luminosidades = datos['luminosidades']
            self.distancias = datos['distancias']
        print("Datos restaurados correctamente desde NPZ.")

    # Formato Pickle
    def guardar_pickle(self):
        # Guarda un diccionario con todas las columnas en resultados/catalogo_completo.pkl
        self._asegurar_directorio()
        ruta = os.path.join("resultados", "catalogo_completo.pkl")
        diccionario_datos = {
            'ids': self.ids,
            'nombres': self.nombres,
            'temperaturas': self.temperaturas,
            'luminosidades': self.luminosidades,
            'distancias': self.distancias
        }
        with open(ruta, 'wb') as archivo:
            pickle.dump(diccionario_datos, archivo)
        print(f"Diccionario de datos guardado en Pickle: {ruta}")

    def cargar_pickle(self):
        # Recupera el diccionario de columnas desde el archivo Pickle
        ruta = os.path.join("resultados", "catalogo_completo.pkl")
        with open(ruta, 'rb') as archivo:
            diccionario_datos = pickle.load(archivo)
        self.ids = diccionario_datos['ids']
        self.nombres = diccionario_datos['nombres']
        self.temperaturas = diccionario_datos['temperaturas']
        self.luminosidades = diccionario_datos['luminosidades']
        self.distancias = diccionario_datos['distancias']
        print("Datos restaurados correctamente desde Pickle.")

    def verificar_archivos(self):
        print("===============VERIFICACIÓN DE INTEGRIDAD DE DATOS===============")
        # Verificar NPY
        try:
            temp_npy = np.load(os.path.join("resultados", "temperaturas.npy"))
            match_npy = np.allclose(self.temperaturas, temp_npy, equal_nan=True)
            print(f"[-] Archivo NPY (Temperaturas) Coincide: {match_npy}")
        except Exception as e:
            print(f"[X] Error al verificar NPY: {e}")
        # Verificar CSV
        try:
            ids_csv, nom_csv, temp_csv, lum_csv, dist_csv = [], [], [], [], []
            with open(os.path.join("resultados", "catalogo_copia.csv"), mode='r', encoding='utf-8') as f:
                lector = csv.DictReader(f)
                for fila in lector:
                    ids_csv.append(fila['id'])
                    nom_csv.append(fila['nombre'])
                    temp_csv.append(float(fila['temperatura']) if fila['temperatura'] else np.nan)
                    lum_csv.append(float(fila['luminosidad']) if fila['luminosidad'] else np.nan)
                    dist_csv.append(float(fila['distancia']) if fila['distancia'] else np.nan)
            match_id = np.array_equal(self.ids, np.array(ids_csv))
            match_num = np.allclose(self.temperaturas, np.array(temp_csv), equal_nan=True) and \
                        np.allclose(self.luminosidades, np.array(lum_csv), equal_nan=True) and \
                        np.allclose(self.distancias, np.array(dist_csv), equal_nan=True)
            print(f"[-] Archivo CSV Coincide: {match_id and match_num}")
        except Exception as e:
            print(f"[X] Error al verificar CSV: {e}")
        # Verificar NPZ
        try:
            with np.load(os.path.join("resultados", "catalogo_completo.npz"), allow_pickle=False) as datos:
                match_id = np.array_equal(self.ids, datos['ids'])
                match_nom = np.array_equal(self.nombres, datos['nombres'])
                match_num = np.allclose(self.temperaturas, datos['temperaturas'], equal_nan=True) and \
                            np.allclose(self.luminosidades, datos['luminosidades'], equal_nan=True) and \
                            np.allclose(self.distancias, datos['distancias'], equal_nan=True)
            print(f"[-] Archivo NPZ (Comprimido) Coincide: {match_id and match_nom and match_num}")
        except Exception as e:
            print(f"[X] Error al verificar NPZ: {e}")
        # Verificar Pickle
        try:
            with open(os.path.join("resultados", "catalogo_completo.pkl"), 'rb') as f:
                datos = pickle.load(f)
                match_id = np.array_equal(self.ids, datos['ids'])
                match_nom = np.array_equal(self.nombres, datos['nombres'])
                match_num = np.allclose(self.temperaturas, datos['temperaturas'], equal_nan=True) and \
                            np.allclose(self.luminosidades, datos['luminosidades'], equal_nan=True) and \
                            np.allclose(self.distancias, datos['distancias'], equal_nan=True)
            print(f"[-] Archivo Pickle (.pkl) Coincide: {match_id and match_nom and match_num}")
        except Exception as e:
            print(f"[X] Error al verificar Pickle: {e}")

    def comparar_formatos(self):
        archivos = {
            "CSV": "catalogo_copia.csv",
            "NPY (Solo temperaturas)": "temperaturas.npy",
            "NPZ": "catalogo_completo.npz",
            "Pickle": "catalogo_completo.pkl"
        }
        print(f"{'Formato de Archivo':<28} | {'Tamaño (Bytes)':<15} | {'Tamaño (KB)':<10}")
        for nombre, archivo in archivos.items():
            ruta = os.path.join("resultados", archivo)
            if os.path.exists(ruta):
                tamaño_bytes = os.path.getsize(ruta)
                tamaño_kb = tamaño_bytes / 1024
                print(f"{nombre:<28} | {tamaño_bytes:<15,} | {tamaño_kb:<10.2f}")
            else:
                print(f"{nombre:<28} | {'No encontrado':<15} | {'-':<10}")

"""
1. Principales diferencias entre los formatos
CSV: Almacena los datos codificados en texto separados por comas, es legible por cualquier software y por humanos, pero consume mucho espacio.
NPY: Está optimizado para almacenar un único arreglo, es rapido y preciso.
NPZ: Es un archivo contenedor que empaqueta múltiples archivos .npy, y reduce el espacio en el disco sin perder precisión numérica.
Pickle: Transforma diccionarios y listas en flujos de bytes binarios, es flexible y preserva la estructura de objetos pero es exclusivo de Python.
2. ¿Qué formato utilizaría para compartir el catálogo con otros usuarios?
El formato CSV, porque se puede usar en cualquier software (no es exclusivo) y se puede leer directamente.
3. ¿Cuál considera más apropiado para almacenar varios arreglos NumPy en un único archivo?
El formato NPZ, porque está hecho en base a Numpy y es para almacenar varios arreglos.
"""

'\n1. Principales diferencias entre los formatos\nCSV: Almacena los datos codificados en texto separados por comas, es legible por cualquier software y por humanos, pero consume mucho espacio.\nNPY: Está optimizado para almacenar un único arreglo, es rapido y preciso.\nNPZ: Es un archivo contenedor que empaqueta múltiples archivos .npy, y reduce el espacio en el disco sin perder precisión numérica.\nPickle: Transforma diccionarios y listas en flujos de bytes binarios, es flexible y preserva la estructura de objetos pero es exclusivo de Python.\n2. ¿Qué formato utilizaría para compartir el catálogo con otros usuarios?\nEl formato CSV, porque se puede usar en cualquier software (no es exclusivo) y se puede leer directamente.\n3. ¿Cuál considera más apropiado para almacenar varios arreglos NumPy en un único archivo?\nEl formato NPZ, porque está hecho en base a Numpy y es para almacenar varios arreglos.\n'

In [43]:
mi_catalogo = CatalogoEstelar("catalogo_estelar.csv") # Para guardar la clase con el catalogo y usar sus funciones